<a href="https://colab.research.google.com/github/arblot/Drone-Tracker/blob/main/Neat_vs_Gokul.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install qiskit-ibm-runtime qiskit-aer
from qiskit import qpy, transpile
from qiskit.quantum_info import SparsePauliOp, Clifford
from qiskit_ibm_runtime.debug_tools import Neat
from qiskit_aer import AerSimulator
from qiskit.circuit.quantumcircuit import QuantumCircuit
from math import isclose, pi
import numpy as np
from qiskit_ibm_runtime.fake_provider import FakeBrisbane

In [ ]:
with open('/content/50c10_sc0_transpiled_circuit_05012026.qpy', 'rb') as f:
    qc = qpy.load(f)[0]

In [ ]:
def remove_idle_qubits(circ):
    """Rebuilds a circuit using only the qubits that have operations on them."""
    # 1. Identify which qubits are actually being used
    active_qubits = set()
    for instruction in circ.data:
        for qubit in instruction.qubits:
            active_qubits.add(qubit)

    # 2. Maintain original order for the active qubits
    active_qubit_list = [q for q in circ.qubits if q in active_qubits]

    # 3. Create a fresh circuit with the correct, smaller number of qubits
    clean_circ = QuantumCircuit(len(active_qubit_list), circ.num_clbits)

    # 4. Create a mapping dictionary: Old Qubit -> New Qubit
    qubit_mapping = {old_q: clean_circ.qubits[i] for i, old_q in enumerate(active_qubit_list)}
    clbit_mapping = {old_c: clean_circ.clbits[i] for i, old_c in enumerate(circ.clbits)}

    # 5. Copy all operations over using the new wiring
    for instruction in circ.data:
        new_qargs = [qubit_mapping[q] for q in instruction.qubits]
        new_cargs = [clbit_mapping[c] for c in instruction.clbits]
        clean_circ.append(instruction.operation, new_qargs, new_cargs)

    return clean_circ

# Clean up the circuits!
qc = remove_idle_qubits(qc)
print(f"Original QC - Original Wires: {qc.num_qubits}")
print(f"Original QC - Active Wires:   {qc.num_qubits}")

Original QC - Original Wires: 50
Original QC - Active Wires:   50


In [ ]:
observable = SparsePauliOp("Z" + "I" * (qc.num_qubits - 1))
fake_backend = FakeBrisbane()
noisy_simulator = AerSimulator.from_backend(fake_backend)
#backend = AerSimulator()
neat = Neat(backend=noisy_simulator)
pub = (qc, observable)
ibm_cliff_circ = neat.to_clifford([pub])[0].circuit

In [ ]:
# Gokul Code

HALF_PI = pi / 2
ANGLE_ATOL = 1e-9


def modify_gates(circuit: QuantumCircuit) -> QuantumCircuit:
    qc = QuantumCircuit(circuit.num_qubits, circuit.num_clbits)
    for item in circuit:
        if item.operation.name == "rz":
            angle = float(item.operation.params[0])
            quarter_turns = round(angle / HALF_PI)
            snapped_angle = quarter_turns * HALF_PI
            if not isclose(angle, snapped_angle, abs_tol=ANGLE_ATOL):
                raise ValueError(
                    f"angle is not multiple of pi/2, instead it is {angle}"
                )

            normalized_turns = int(quarter_turns) % 4
            qubit = circuit.find_bit(item.qubits[0]).index
            if normalized_turns == 1:
                qc.s(qubit)
            elif normalized_turns == 2:
                qc.z(qubit)
            elif normalized_turns == 3:
                qc.s(qubit)
                qc.z(qubit)
        else:
            mapped_qubits = [qc.qubits[circuit.find_bit(q).index] for q in item.qubits]
            mapped_clbits = [qc.clbits[circuit.find_bit(c).index] for c in item.clbits]
            qc.append(item.operation.copy(), mapped_qubits, mapped_clbits)
    return qc


def _round_to_closest_angle(angle: float) -> float:
    return round(angle / HALF_PI) * HALF_PI


def to_clifford(qc: QuantumCircuit) -> QuantumCircuit:
    """
    Round the rotation gates to the closest angle of pi/2, then convert the rotation gates to finite sets of gates, s, z
    """
    new_qc = QuantumCircuit(qc.num_qubits, qc.num_clbits)
    for i in qc:
        new_operation = i.operation.copy()
        if i.operation.name == "rz":
            new_operation.params[0] = _round_to_closest_angle(
                float(i.operation.params[0])
            )
        mapped_qubits = [new_qc.qubits[qc.find_bit(q).index] for q in i.qubits]
        mapped_clbits = [new_qc.clbits[qc.find_bit(c).index] for c in i.clbits]
        new_qc.append(new_operation, mapped_qubits, mapped_clbits)
    return modify_gates(new_qc)



qcc1 = to_clifford(qc)
print("clifford circuit:")
#print(qcc)
print(qcc1 != qc)


clifford circuit:
True


In [ ]:
# Compare logic of Gokul vs Neat
print(f"Are they logically identical? {Clifford(qcc1) == Clifford(ibm_cliff_circ)}")

Are they logically identical? True


In [ ]:
# Compare depth / gates of all circuits

print("Original Circuit Ops:", qc.count_ops())
print("Original Circuit Depth:", qc.depth())

print("Gokul Circuit Ops:", qcc1.count_ops())
print("Gokul Circuit Depth:", qcc1.depth())

print("NEAT Circuit Ops:", ibm_cliff_circ.count_ops())
print("NEAT Circuit Depth:", ibm_cliff_circ.depth())

Original Circuit Ops: OrderedDict({'sx': 14976, 'rz': 9898, 'cz': 7991, 'x': 254})
Original Circuit Depth: 6159
Gokul Circuit Ops: OrderedDict({'sx': 14976, 'cz': 7991, 'z': 7019, 's': 5596, 'x': 254})
Gokul Circuit Depth: 6888
NEAT Circuit Ops: OrderedDict({'sx': 14976, 'rz': 9898, 'cz': 7991, 'x': 254})
NEAT Circuit Depth: 6159


In [ ]:
mapped_ibm = transpile(ibm_cliff_circ, backend=fake_backend, optimization_level=0)
mapped_observable = observable.apply_layout(mapped_ibm.layout)

pub_ibm = (mapped_ibm, mapped_observable)
pubs = [pub_ibm]

ideal_results = neat.ideal_sim(pubs, cliffordize=True)
noisy_results = neat.noisy_sim(pubs, cliffordize=True)


ideal = np.atleast_1d(ideal_results.vals)[0]
noisy = np.atleast_1d(noisy_results.vals)[0]

absolute_error = abs(ideal - noisy)

print(f"  Ideal Expectation: {ideal:.4f}")
print(f"  Noisy Expectation: {noisy:.4f}")
print(f"  Absolute Error:    {absolute_error:.4f}")

if ideal != 0:
    signal_remaining = noisy / ideal
    signal_loss = (1.0 - signal_remaining) * 100
    print(f"  Signal Lost:       {signal_loss:.2f}%")
else:
    print("  Signal Lost:       Cannot compute % (Ideal is 0)")